In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import os

df = pd.read_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/upi_transactions_2024.csv')
print("Shape:", df.shape)
print("Fraud cases:", df['fraud_flag'].sum())

Shape: (250000, 17)
Fraud cases: 480


In [2]:
drop_cols = ['transaction id', 'timestamp', 'transaction_status']

df = df.drop(columns=drop_cols)
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['transaction type', 'merchant_category', 'amount (INR)', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']


In [3]:
# Flag for night transactions (highest fraud hours from EDA)
df['is_night'] = df['hour_of_day'].apply(lambda x: 1 if x <= 4 else 0)

# Flag for high amount transactions (above 75th percentile = ₹1596)
df['is_high_amount'] = df['amount (INR)'].apply(lambda x: 1 if x > 1596 else 0)

# Amount bucketed into bands (more useful than raw amount for tree models)
df['amount_band'] = pd.cut(df['amount (INR)'],
                            bins=[0, 500, 1500, 5000, 50000],
                            labels=['low', 'medium', 'high', 'very_high'])

print("New features added: is_night, is_high_amount, amount_band")
print(df[['amount (INR)', 'hour_of_day', 'is_night', 'is_high_amount', 'amount_band']].head(8))

New features added: is_night, is_high_amount, amount_band
   amount (INR)  hour_of_day  is_night  is_high_amount amount_band
0           868           15         0               0      medium
1          1011            6         0               0      medium
2           477           13         0               0         low
3          2784           10         0               1        high
4           990           19         0               0      medium
5            91           22         0               0         low
6           314           10         0               0         low
7           264           18         0               0         low


In [5]:
# Columns that need converting from text to numbers
categorical_cols = [
    'transaction type',
    'merchant_category',
    'sender_age_group',
    'receiver_age_group',
    'sender_state',
    'sender_bank',
    'receiver_bank',
    'device_type',
    'network_type',
    'day_of_week',
    'amount_band'
]

le = LabelEncoder()
label_encoders = {}  # Save encoders to reuse during prediction

for col in categorical_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"Encoded '{col}' → {df[col].unique()}")

# Save encoders for use in the API later
os.makedirs('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model', exist_ok=True)
joblib.dump(label_encoders, 'D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model/label_encoders.pkl')
print("\nLabel encoders saved.")

Encoded 'transaction type' → [2 1 0 3]
Encoded 'merchant_category' → [1 4 3 7 2 6 9 8 5 0]
Encoded 'sender_age_group' → [1 2 3 4 0]
Encoded 'receiver_age_group' → [0 1 2 3 4]
Encoded 'sender_state' → [1 8 3 7 4 2 5 6 9 0]
Encoded 'sender_bank' → [0 2 7 3 1 4 6 5]
Encoded 'receiver_bank' → [6 0 5 7 3 1 4 2]
Encoded 'device_type' → [0 2 1]
Encoded 'network_type' → [1 2 3 0]
Encoded 'day_of_week' → [5 4 3 1 2 6 0]
Encoded 'amount_band' → [2 1 0 3]

Label encoders saved.


In [6]:
print("=== Final column list ===")
print(df.columns.tolist())

print("\n=== Data types ===")
print(df.dtypes)

print("\n=== Any nulls? ===")
print(df.isnull().sum().sum(), "null values remaining")

print("\n=== Sample rows ===")
df.head()

=== Final column list ===
['transaction type', 'merchant_category', 'amount (INR)', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'is_high_amount', 'amount_band']

=== Data types ===
transaction type      int64
merchant_category     int64
amount (INR)          int64
sender_age_group      int64
receiver_age_group    int64
sender_state          int64
sender_bank           int64
receiver_bank         int64
device_type           int64
network_type          int64
fraud_flag            int64
hour_of_day           int64
day_of_week           int64
is_weekend            int64
is_night              int64
is_high_amount        int64
amount_band           int64
dtype: object

=== Any nulls? ===
0 null values remaining

=== Sample rows ===


,transaction type,merchant_category,amount (INR),sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,is_night,is_high_amount,amount_band
0,2,1,868,1,0,1,0,6,0,1,0,15,5,0,0,0,2
1,1,4,1011,1,1,8,2,0,2,1,0,6,4,0,0,0,2
2,2,4,477,1,2,3,7,5,0,1,0,13,5,0,0,0,1
3,2,3,2784,1,1,1,2,5,0,2,0,10,3,1,0,1,0
4,2,7,990,1,0,1,0,7,2,3,0,19,5,0,0,0,2


In [7]:
X = df.drop(columns=['fraud_flag'])  
y = df['fraud_flag']                 

print("Features shape:", X.shape)    
print("Target shape:", y.shape)      
print("Fraud cases in y:", y.sum())  

Features shape: (250000, 16)
Target shape: (250000,)
Fraud cases in y: 480


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        
    random_state=42,      
    stratify=y            
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Fraud in train:", y_train.sum())
print("Fraud in test:", y_test.sum())

Training set: (200000, 16)
Test set: (50000, 16)
Fraud in train: 384
Fraud in test: 96


In [9]:
print("Before SMOTE:")
print("Legitimate:", sum(y_train == 0))
print("Fraud:", sum(y_train == 1))

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print("Legitimate:", sum(y_train_sm == 0))
print("Fraud:", sum(y_train_sm == 1))

Before SMOTE:
Legitimate: 199616
Fraud: 384

After SMOTE:
Legitimate: 199616
Fraud: 199616


In [10]:
scale_cols = ['amount (INR)', 'hour_of_day']

scaler = StandardScaler()
X_train_sm[scale_cols] = scaler.fit_transform(X_train_sm[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])  


joblib.dump(scaler, 'D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model/scaler.pkl')
print("Scaler saved.")
print(X_train_sm[scale_cols].describe().round(3))

Scaler saved.
       amount (INR)  hour_of_day
count    399232.000   399232.000
mean         -0.000        0.000
std           1.000        1.000
min          -0.672       -3.042
25%          -0.546       -0.757
50%          -0.383        0.074
75%           0.113        0.697
max          19.590        1.736


In [11]:
import os
os.makedirs('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/processed', exist_ok=True)

X_train_sm.to_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/processed/X_train.csv', index=False)
X_test.to_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/processed/X_test.csv', index=False)
y_train_sm.to_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/processed/y_train.csv', index=False)
y_test.to_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/processed/y_test.csv', index=False)

print("All processed files saved to data/processed/")
print("X_train shape:", X_train_sm.shape)
print("X_test shape:", X_test.shape)

All processed files saved to data/processed/
X_train shape: (399232, 16)
X_test shape: (50000, 16)
